In [58]:
# --- ⚙️ 모델 설정 구역 ---
# [중요] EMBEDDING_MODEL을 변경하면 반드시 '1_make_embedding.ipynb'를 실행하여 
# 새로운 모델에 맞는 .pkl 파일을 다시 생성해야 합니다. (차원이 다르면 FAISS 에러 발생)
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

# 모델 이름에 따라 차원을 자동으로 정해주는 딕셔너리(사전)를 만듭니다.
MODEL_DIMENSIONS = {
    "text-embedding-3-small": 1536,
    "text-embedding-3-large": 3072,
    "text-embedding-ada-002": 1536
}

# 현재 선택된 모델에 맞는 차원 값을 가져옵니다.
# (만약 사전에 없는 모델이면 기본값으로 1536을 씁니다.)
dimension = MODEL_DIMENSIONS.get(EMBEDDING_MODEL, 1536)

print(f"✅ 현재 설정된 모델: {EMBEDDING_MODEL}")
print(f"📏 적용된 차원(Dimension): {dimension}")

✅ 현재 설정된 모델: text-embedding-3-small
📏 적용된 차원(Dimension): 1536


In [81]:
# API 키 테스트 코드
import os
from dotenv import load_dotenv
from openai import OpenAI

# 1. .env 파일에 적힌 비밀번호들을 불러옵니다.
load_dotenv()

# 2. 시스템 환경 변수에서 'OPENAI_API_KEY'라는 이름의 값을 꺼내옵니다.
# (이때 .env 파일에 적은 왼쪽 이름과 똑같아야 합니다!)
MY_API_KEY = os.getenv("OPENAI_API_KEY") # 본인 .env 파일에 적은 변수명으로 수정하세요.

# 3. 이제 안전하게 불러온 키를 사용해 클라이언트를 만듭니다.
client = OpenAI(api_key=MY_API_KEY)

try:
    # 아주 짧은 단어 하나만 임베딩 시도
    response = client.embeddings.create(input=["hi"], model=EMBEDDING_MODEL)
    print("✅ API 키 인증 성공!")
except Exception as e:
    print(f"❌ API 키 오류: {e}")
    print("팁: 카드 결제 정보나 잔액(Credit)이 있는지 확인해 보세요.")

✅ API 키 인증 성공!


In [73]:
import pandas as pd
import numpy as np
import faiss
from openai import OpenAI

client = OpenAI(api_key=MY_API_KEY)
df = pd.read_pickle('./bid_master_optimized_v2.pkl')

# 2. FAISS 인덱스 구축
print(f"⏳ {EMBEDDING_MODEL} 모델에 맞춰 FAISS 인덱스를 생성 중입니다...")

# [수정] 숫자 1536 대신, 위에서 정의한 변수 'dimension'을 넣습니다!
index = faiss.IndexFlatL2(dimension) 

embeddings_matrix = np.vstack(df['embedding'].values).astype('float32')
index.add(embeddings_matrix)
print(f"✅ FAISS 준비 완료! (차원: {dimension}, 데이터: {index.ntotal}건)")

# 3. FAISS 기반 Naive Retrieval 함수 (top-k)
def search_faiss(query_text, top_k=5):
    # (1) 질문 임베딩화
    query_response = client.embeddings.create(
        input=[query_text], model=EMBEDDING_MODEL
    )
    query_vector = np.array([query_response.data[0].embedding]).astype('float32')
    
    # (2) FAISS 검색 (D: 거리 점수, I: 데이터 인덱스 번호)
    distances, indices = index.search(query_vector, top_k)
    
    # (3) 검색된 인덱스로 데이터 추출
    results = df.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    return results

# 4. FAISS + Pandas 필터링 고도화 함수 (1억 이상 조건 등)
def search_faiss_pro(query_text, top_k=5, min_budget=None):
    # FAISS는 필터링 기능이 없으므로, 먼저 넉넉하게(예: 20개) 뽑습니다.
    candidate_results = search_faiss(query_text, top_k=20)
    
    # 그 후 Pandas를 이용해 메타데이터 조건을 수동으로 겁니다.
    if min_budget:
        candidate_results = candidate_results[candidate_results['사업 금액'] >= min_budget]
    
    return candidate_results.head(top_k)

# 5. FAISS 기반 챗봇 함수 (대화 히스토리 포함)
chat_memory = []

def ask_bid_chatbot_faiss(query, min_budget=None):
    global chat_memory
    
    # 검색 및 필터링
    search_results = search_faiss_pro(query, top_k=3, min_budget=min_budget)
    
    context = ""
    for _, row in search_results.iterrows():
        context += f"사업명: {row['사업명']}\n상세내용: {row['청크_텍스트']}\n---\n"

    # 메시지 구성 (과거 대화 포함)
    messages = [{"role": "system", "content": "입찰 공고 전문가로서 컨텍스트를 바탕으로 답해줘."}]
    for hist in chat_memory[-3:]:
        messages.append({"role": "user", "content": hist['q']})
        messages.append({"role": "assistant", "content": hist['a']})
    messages.append({"role": "user", "content": f"컨텍스트:\n{context}\n\n질문: {query}"})

    response = client.chat.completions.create(
        model=LLM_MODEL, messages=messages, temperature=0
    )
    
    answer = response.choices[0].message.content
    chat_memory.append({"q": query, "a": answer})
    return answer

⏳ text-embedding-3-small 모델에 맞춰 FAISS 인덱스를 생성 중입니다...
✅ FAISS 준비 완료! (차원: 1536, 데이터: 965건)


In [74]:
import faiss
import os
import numpy as np

index_file = "bid_index.faiss"

# 1. 파일이 존재하는지 먼저 확인
if os.path.exists(index_file):
    # 일단 불러와서 개수를 확인해봅니다.
    temp_index = faiss.read_index(index_file)
    
    # [핵심] 현재 df의 데이터 개수와 파일의 인덱스 개수가 같은지 비교!
    if temp_index.ntotal == len(df):
        index = temp_index
        print(f"🚀 기존 인덱스 파일을 불러왔습니다. (총 {index.ntotal}건 - 현재 데이터와 일치)")
    else:
        # 개수가 다르면 (예: 1199 vs 965) 삭제하고 새로 만듭니다.
        print(f"⚠️ 데이터 개수가 달라졌습니다! ({temp_index.ntotal} -> {len(df)})")
        print("🗑️ 구형 인덱스 파일을 삭제하고 새로 생성합니다...")
        os.remove(index_file)
        index = None
else:
    index = None

# 2. 인덱스가 없거나 새로 만들어야 하는 경우
if index is None:
    print(f"⏳ 새 FAISS 인덱스를 생성 중입니다... (데이터: {len(df)}건)")
    
    # 인덱스 초기화
    index = faiss.IndexFlatL2(dimension)
    
    # 임베딩 추가
    embeddings_matrix = np.vstack(df['embedding'].values).astype('float32')
    index.add(embeddings_matrix)
    
    # 파일로 저장
    faiss.write_index(index, index_file)
    print(f"✅ 새 인덱스 생성 및 '{index_file}' 저장 완료! (총 {index.ntotal}건)")

🚀 기존 인덱스 파일을 불러왔습니다. (총 965건 - 현재 데이터와 일치)


In [78]:
import pandas as pd

# 1. 전역 저장소 생성 (기록 및 팩트체크용)
if 'chat_memory' not in globals():
    chat_memory = []

# 방금 답변의 근거 데이터를 임시 보관할 변수
last_retrieved_context = [] 

# 2. 수빈님의 전문가 프롬프트 정의
SYSTEM_PROMPT = """당신은 B2G 공공입찰 전문 컨설팅 어시스턴트입니다.
주어진 RFP 문서 내용만을 기반으로 답변하세요.
문서에 없는 내용은 반드시 "해당 문서에서 확인할 수 없습니다"라고 답하세요.
답변은 구조화된 형식으로 제공하고, 출처 문서명을 항상 명시하세요."""

# 3. 재정렬(Re-ranking) 함수
def re_rank(query, results_list):
    query_words = [word for word in query.split() if len(word) > 1]
    results_df = pd.DataFrame(results_list)

    def calculate_score(text):
        return sum(1 for word in query_words if word.lower() in str(text).lower())

    results_df['re_rank_score'] = results_df['청크_텍스트'].apply(calculate_score)
    sorted_df = results_df.sort_values(by='re_rank_score', ascending=False)
    return sorted_df

# 4. 통합 챗봇 함수 (v3 - 팩트체크 데이터 저장 기능 추가)
def ask_bid_chatbot_v3(query):
    global chat_memory, last_retrieved_context
    
    # (1) FAISS 검색 및 Re-ranking
    search_results = search_faiss_pro(query, top_k=5)
    sorted_results = re_rank(query, search_results)
    
    # [핵심] 나중에 다른 셀에서 확인할 수 있도록 검색 결과 3개를 저장합니다.
    last_retrieved_context = sorted_results.head(3).to_dict('records')
    
    # (2) 컨텍스트 생성 (출처 정보 포함)
    context = ""
    for _, row in sorted_results.head(3).iterrows():
        biz_name = row.get('사업명', '알 수 없는 사업')
        bid_no = row.get('공고번호', '정보 없음')
        source_info = f"[출처: {biz_name} | 공고번호: {bid_no}]"
        context += f"{source_info}\n{row['청크_텍스트']}\n---\n"

    # (3) 대화 히스토리 정리
    history_text = ""
    for hist in chat_memory[-3:]:
        history_text += f"사용자: {hist['q']}\n어시스턴트: {hist['a']}\n"

    # (4) 최종 프롬프트 구성
    final_user_content = f"""
[참고 문서]
{context}
[대화 히스토리]
{history_text if history_text else "없음"}
[질문]
{query}
위 문서를 기반으로 정확하게 답변하세요. 출처를 반드시 포함하세요.
"""

    # (5) OpenAI LLM 호출
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": final_user_content}
        ],
        temperature=0
    )
    
    answer = response.choices[0].message.content
    chat_memory.append({"q": query, "a": answer})
    
    return answer

In [79]:
# 새로운 사업 분석을 시작하거나, 대화가 꼬였다면 아래 줄의 주석을 풀고 실행하세요.
chat_memory = [] 
last_retrieved_context = [] # 팩트체크용 저장소도 같이 비워주는 게 좋습니다.

print("✅ 대화 기록이 초기화되었습니다. (필요 시에만 실행)")

✅ 대화 기록이 초기화되었습니다. (필요 시에만 실행)


In [80]:
while True:
    user_input = input("❓ 질문 입력 (종료: q, 근거확인: 팩트): ").strip()
    
    if user_input.lower() in ['q', 'quit', '종료']:
        print("👋 대화를 종료합니다.")
        break
        
    # [추가] '팩트'라고 입력하면 저장된 마지막 근거를 보여줌
    if user_input == "팩트":
        if not last_retrieved_context:
            print("❌ 아직 진행된 대화가 없습니다.")
        else:
            print(f"\n🔎 [실시간 팩트체크] 방금 답변의 근거입니다.")
            for i, row in enumerate(last_retrieved_context):
                print(f"[{i+1}번] {row.get('사업명')} | 점수: {row.get('re_rank_score')}")
                print(f"📄 원문: {row.get('청크_텍스트')[:300]}...\n")
        continue # 근거 보여준 뒤 다시 질문 입력으로 돌아감
    
    if not user_input: continue

    print(f"\n🔍 분석 중...")
    answer = ask_bid_chatbot_v3(user_input) # 일반 버전 호출
    
    print(f"\n🤖 AI 답변:\n{answer}")
    print("\n" + "="*60)

❓ 질문 입력 (종료: q, 근거확인: 팩트):  데이터 분석을 바탕으로 한 의사결정 지원 시스템이나 자동화 솔루션 도입 사업이 있어?



🔍 분석 중...

🤖 AI 답변:
해당 문서에서 확인할 수 없습니다.



❓ 질문 입력 (종료: q, 근거확인: 팩트):  모바일 앱 구축 경험이 있는 고객사에게 적합한 사업을 추천해줘.



🔍 분석 중...

🤖 AI 답변:
모바일 앱 구축 경험이 있는 고객사에게 적합한 사업은 다음과 같습니다:

### 1. EIP3.0 고압가스 안전관리 시스템 구축 용역
- **발주기관**: 한국생산기술연구원
- **사업명**: EIP3.0 고압가스 안전관리 시스템 구축 용역
- **금액**: 40,000,000원
- **시작일**: 2024-08-29
- **요구사항**:
  - 다양한 화면 해상도의 스마트폰과 태블릿에서 사용 가능한 모바일 사용자 인터페이스 개발
  - 모바일 서비스 개발 관련 지침 준수
  - UI/UX 설계 표준 정의 및 사용자 편의성을 고려한 직관적인 UI 개발

### 2. 모바일오피스 시스템 고도화 용역
- **발주기관**: 한국철도공사
- **사업명**: 모바일오피스 시스템 고도화 용역(총체 및 1차)
- **금액**: 843,000,000원
- **시작일**: 2024-10-18
- **요구사항**:
  - 사용자 중심의 UI/UX 신규 설계 및 적용
  - 직관적인 UI/UX로 편의성 개선
  - 하이브리드 앱으로 전환 구축하여 앱 용량 및 재배포 횟수 축소
  - 기존 업무전용앱 기능을 모바일오피스에 통합

이 두 사업은 모바일 앱 구축 경험이 있는 고객사에게 적합하며, 사용자 편의성과 최신 기술을 반영한 UI/UX 설계가 요구됩니다.

**출처**: 
- EIP3.0 고압가스 안전관리 시스템 구축 용역 | 공고번호: 20240827859
- 모바일오피스 시스템 고도화 용역(총체 및 1차) | 공고번호: TEMP_37



❓ 질문 입력 (종료: q, 근거확인: 팩트):  팩트



🔎 [실시간 팩트체크] 방금 답변의 근거입니다.
[1번] EIP3.0 고압가스 안전관리 시스템 구축 용역 | 점수: 3
📄 원문: [공고번호: 20240827859 | 발주기관: 한국생산기술연구원 | 사업명: EIP3.0 고압가스 안전관리 시스템 구축 용역 | 금액: 40,000,000원 | 시작일: 2024-08-29 09:00:00 | 조각순서: 15/76]

요구사항 명칭 가 요구사항 상세설명 모바일 사용자 인터페이스 다양한 화면 해상도의 스마트폰과 태블릿에서 공용으로 사용 가능하도록 화면 구성 모바일 서비스 개발 관련 지침 준수 - 모바일 전자정부 서비스 관리 지침 (행정자치부 예규 제25호(2015.8.28.)) - 모바일 전자정부 사용자 인터페이스 ...

[2번] 수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰 공고 | 점수: 3
📄 원문: [공고번호: 20240605067 | 발주기관: 수협중앙회 | 사업명: 수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰 공고 | 금액: 300,000,000원 | 시작일: 2024-06-05 10:02:59 | 조각순서: 6/9]

. 추진 방향 [수산물 사이버직매장 재구축 시스템 서비스 구성도] 이용자 편의성 개선 (개인화 마케팅) AI를 활용한 고객의 구매 행동 및 선호도를 분석하여 개인화된 마케팅 전략 수립 (신속 결제시스템) 결제과정의 단순화 및 결제방식 다양화 - 주요 고객 연령층에 적합한 단순하고 편리한 결...

[3번] 모바일오피스 시스템 고도화 용역(총체 및 1차) | 점수: 2
📄 원문: [공고번호: TEMP_37 | 발주기관: 한국철도공사 (용역) | 사업명: 모바일오피스 시스템 고도화 용역(총체 및 1차) | 금액: 843,000,000원 | 시작일: 2024-10-18 00:00:00 | 조각순서: 4/22]

오류로 앱 중지-종료 현상 빈번히 발생 모바일오피스와 업무전용앱이 이원화 운영되고 있어 관리-유지보수, 필요기능 사용 불편 운영서버 내구연한 초과 및

❓ 질문 입력 (종료: q, 근거확인: 팩트):  q


👋 대화를 종료합니다.
